# Week 2: Transform and Test (PySpark)
**LSA Data Engineering Internship, October 2026 · NHS GP Appointments ETL**

## What this notebook does
Week 1 extracted the three raw NHS files. Week 2 **cleans them with PySpark and proves the cleaning is correct**.
This notebook runs the whole Week 2 pipeline from `run_pipeline.py` on Google Colab and records the evidence for the deliverables.

| Step | What happens | Why |
| --- | --- | --- |
| 1 | Mount Google Drive | The raw files live there, and Colab's machine is temporary |
| 2 | Clone the repo, install packages, check Java | Spark needs Java 17, 21 or 25 |
| 3 | Find the project folder, apply the sub-ICB code fix | The code must match what was tested |
| 4 | Run the pipeline (extract, then transform) | Produces clean Parquet, partitioned by month |
| 5 | Run it a second time | Proves the pipeline is idempotent: same result, never doubled |
| 6 | Measure Parquet against CSV size | Required in the README |
| 7 | Run the pytest suite | Proves each cleaning rule holds on the real data |
| 8 | Break two rules on purpose | A test that never fails proves nothing |

## Numbers the pipeline must reproduce
| Dataset | Rows after transform | Total appointments |
| --- | --- | --- |
| Regional | 295,692 (from 596,821) | 742,804,525 |
| National categories | 817,394 | 296,046,770 |
| Actual duration | 137,793 | 167,980,692 |

## Step 1: Mount Google Drive
Colab wipes its machine when the session ends. Raw data is read from Drive, so connect it first. Click **Allow** when asked.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Get the code, install packages, check Java
- `git clone` downloads the project from GitHub (the `rm -rf` first removes any old copy).
- `pip install` adds the packages the pipeline and tests need. PySpark is already available on Colab.
- Spark is a Java program, so `java -version` must show 17, 21 or 25.

In [ ]:
%cd /content
!rm -rf lsa-oct-2026
!git clone https://github.com/deepanshuantil/lsa-oct-2026.git
!pip install -q "pandas<3" pyarrow openpyxl pytest
!java -version
# If Java is missing, uncomment the next line and run this cell again:
# !apt-get -qq install -y openjdk-21-jre-headless

## Step 3: Find the project folder and apply the sub-ICB code fix
**Project folder.** The repo may contain the code at the top level or inside one extra `lsa-oct-2026` folder. This cell finds `config.py` either way and moves into that folder.

**Sub-ICB code fix.** The categories file holds location names such as `NHS North East London ICB - A3A8R`.
The code at the end is **3 to 5 characters** long (95,147 rows have 5-character codes). The original pattern only allowed 3, so the transform stopped on purpose.
This cell changes `{3}` to `{3,5}`. It is safe to run more than once, and it does nothing once the repo already contains the fix.

In [ ]:
import os, pathlib

root = next(p for p in [pathlib.Path('/content/lsa-oct-2026'),
                        pathlib.Path('/content/lsa-oct-2026/lsa-oct-2026')]
            if (p / 'config.py').exists())
os.chdir(root)
print('Working in:', root)

for path in ['src/transform.py', 'tests/test_transform.py']:
    text = open(path).read()
    open(path, 'w').write(text.replace('[0-9A-Z]{3}', '[0-9A-Z]{3,5}'))
!grep -n "0-9A-Z" src/transform.py tests/test_transform.py

## Step 4: Run the pipeline (first run)
`run_pipeline.py` runs two stages in order and stops at the first failure:
1. **Extract** copies the three files into `data/raw/`, converts the Excel file to CSV and writes `manifest.json`.
2. **Transform** (PySpark) reads each file with an **explicit schema**, fixes the five traps (hidden BOM, day-first dates, fake CSV, look-alike rows summed not dropped, messy labels), checks the totals after each step, and writes Parquet partitioned by month.

`LSA_SOURCE` tells the pipeline where the original files are on Drive. Change the path if yours is different.
It takes about 7 minutes. **Success:** `pipeline finished OK` with the three totals above.

In [ ]:
os.environ['LSA_SOURCE'] = '/content/drive/MyDrive/LSA_Oct2026/raw'
!python run_pipeline.py

## Step 5: Run it again to prove it is idempotent
Running the pipeline twice must give **identical** results. Every stage overwrites its own output, so nothing is doubled.
Compare the last log line with Step 4: it must show the same row counts and totals.

In [ ]:
!python run_pipeline.py

## Step 6: Measure storage (for the README)
CSV is how data arrives. Parquet is compressed, columnar and remembers its column types.

In [ ]:
!du -ch data/raw/*.csv | tail -1
!du -sh data/clean

## Step 7: Run the pytest suite
There is at least one test per cleaning rule: totals preserved, row counts, unique keys after summing, tidy labels, one spelling of *Other Practice Staff*, no byte-order mark, day-first dates, sub-ICB codes, positive whole-number counts, month partitioning, and an identical second run.
The tests run the transform themselves, so this takes about 15 minutes. **Success:** all tests passed, none skipped.

In [ ]:
!python -m pytest tests -q

## Step 8: Break the rules on purpose
A test that never fails proves nothing. Each cell below breaks one rule in `transform.py`, runs the tests, and then **restores the original file**.
Each takes about 8 minutes because the tests re-run the transform.

**Break 1: stop unifying "Other Practice staff" / "Other Practice Staff".** Expected: `test_one_spelling_of_other_practice_staff` FAILS.

In [ ]:
import shutil
shutil.copy('src/transform.py', '/content/backup.py')
s = open('src/transform.py').read()
open('src/transform.py', 'w').write(
    s.replace('F.lower(F.col("hcp_type")) == "other practice staff"', 'F.lit(False)'))
!python -m pytest tests -q -x 2>&1 | tail -5
shutil.copy('/content/backup.py', 'src/transform.py')   # restore the original

**Break 2: stop partitioning the Parquet by month.** Expected: `test_parquet_is_partitioned_by_month` FAILS.

In [ ]:
s = open('src/transform.py').read()
open('src/transform.py', 'w').write(s.replace('.partitionBy("month_start")', ''))
!python -m pytest tests -q -x 2>&1 | tail -5
shutil.copy('/content/backup.py', 'src/transform.py')   # restore the original

**Confirm the original file is back.** This must print `1`.

In [ ]:
!grep -c partitionBy src/transform.py

## Results (recorded from the real run)
| Check | Result |
| --- | --- |
| Pipeline, first run | Finished OK, about 7 minutes |
| Regional | 596,821 to **295,692** rows (301,129 look-alike rows merged), total **742,804,525** |
| Categories | 817,394 rows, total **296,046,770** |
| Duration | 137,793 rows, total **167,980,692** |
| Pipeline, second run | Identical totals |
| Storage | Source CSVs **159 MB** to clean Parquet **5.6 MB** |
| pytest | **23 passed**, none skipped (includes the brief's exact numbers) |
| Break 1 (staff spelling) | `test_one_spelling_of_other_practice_staff` failed, as expected |
| Break 2 (partitioning) | Run the cell above and record which test failed |

## Notes
- The sub-ICB code is 3 to 5 characters, not 3. This was found by profiling the failure and is recorded in `docs/decision_log.md`.
- Raw files are never edited. Rebuild everything from `data/raw/` if something goes wrong.
- Source: NHS England, Appointments in General Practice.